In [2]:
import os
import sys

sys.path.append(os.path.abspath(".."))

In [3]:
from src.llama_style_50m import Transformer, ModelConfig, fineweb_get_data
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

/Users/den/Desktop/portfolio/llama_style_50m/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = torch.accelerator.current_accelerator()

In [6]:
ckpt = torch.load("../ckpts/pretraining_ckpt", map_location="cpu")
state_dict = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}

In [7]:
model = Transformer(ModelConfig())
model.load_state_dict(state_dict)
model.to(device)

Transformer(
  (embeddings): Embedding(32000, 512)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): RMSNorm((512,), eps=None, elementwise_affine=True)
      (gqa): GQA(
        (q_proj): Linear(in_features=512, out_features=512, bias=False)
        (kv_proj): Linear(in_features=512, out_features=512, bias=False)
        (out_proj): Linear(in_features=512, out_features=512, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (norm2): RMSNorm((512,), eps=None, elementwise_affine=True)
      (swiglu): SwiGLU(
        (value_proj): Linear(in_features=512, out_features=1408, bias=False)
        (gate_proj): Linear(in_features=512, out_features=1408, bias=False)
        (out_proj): Linear(in_features=1408, out_features=512, bias=False)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): RMSNorm((512,), eps=None, elementwise_affine=True)
  (out_proj): Linear(in_features=512, out_features=32000, bias=False)
)

In [8]:
bsz = 16
steps = 30

In [10]:
train_ds, _ = fineweb_get_data("../datasets/fineweb")
loader = DataLoader(train_ds, batch_size=bsz, shuffle=True)

In [11]:
total_loss = 0.0
model.eval()

with torch.no_grad():
    progress_bar = tqdm(loader, total=steps)
    for i, (xs, ys) in enumerate(progress_bar):
        if i >= steps:
            break
            
        xs, ys = xs.to(device), ys.to(device)
        
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
            logits = model(xs)
            loss = F.cross_entropy(logits.flatten(0, 1), ys.flatten(0, 1))
            
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

avg_loss = total_loss / steps

100%|██████████| 30/30 [00:39<00:00,  1.31s/it, loss=3.05]


In [12]:
print("Average Loss: ", avg_loss)
print("Perplexity: ", torch.exp(torch.tensor(avg_loss)).item())

Average Loss:  2.979067413012187
Perplexity:  19.669462203979492
